# Train a LiDAR-only person classifier (Direction A)

Loads **real recorded L2 point clouds** (`datasets/lidar_clouds/`), **views them and the DBSCAN clusters inline (Plotly)**, extracts cluster features, and trains a person classifier so the L2 can count people without the camera. Pure numpy + scikit-learn + plotly (no open3d, no GPU) - runs in the container or any Python env.

The provided clouds are an **empty room** (furniture clusters = negatives); positives/labels come from the camera harvest after calibration, or use the synthetic label set to train now.

Config split into **SWITCHES / QUANTITATIVE / QUALITATIVE**.

## 1. Config

In [ ]:
# SWITCHES (on/off)
USE_SYNTHETIC_LABELS = True   # train on synthetic labeled clusters (no people in provided clouds)
SUBTRACT_BACKGROUND  = False  # True: keep only dynamic foreground; False: cluster full cloud
REMOVE_GROUND        = True   # drop the lowest z-band (floor) before clustering
STANDARDIZE          = True
BALANCE_CLASSES      = True
CROSS_VALIDATE       = True
SAVE_MODEL           = True

In [ ]:
# QUANTITATIVE (numbers / knobs) - the geometry knobs tuned during development
ROI_RADIUS         = 10.0   # m, horizontal crop around sensor
ROI_Z              = 3.0    # m, |z| crop
GROUND_PERCENTILE  = 12     # drop points below this z-percentile (floor) if REMOVE_GROUND
VOXEL              = 0.2    # background voxel size (must match record_background)
DBSCAN_EPS         = 0.35   # cluster neighbourhood radius (m)
DBSCAN_MIN_POINTS  = 12
MIN_CLUSTER_POINTS = 25     # discard clusters smaller than this
HEIGHT_MIN         = 0.8    # human-size gate (for weak labels / filtering)
HEIGHT_MAX         = 2.2
FOOTPRINT_MAX      = 0.9
VIEW_FRAME         = 0      # which recorded cloud to view
VIEW_MAX_POINTS    = 15000  # plotly subsample cap
TEST_SPLIT         = 0.2
N_ESTIMATORS       = 200
MAX_DEPTH          = 8
MIN_SAMPLES_LEAF   = 5
RANDOM_SEED        = 0
CV_FOLDS           = 5

In [ ]:
# QUALITATIVE (strings / choices)
CLOUDS_DIR     = 'datasets/lidar_clouds'      # provided .npy frames (x,y,z,intensity,ring)
BACKGROUND_NPZ = '../calib/background.npz'     # from record_background.py
HARVEST_FILE   = '../runs/harvest.jsonl'       # real labels from fuse.py (post-calibration)
CLASSIFIER     = 'random_forest'               # random_forest | logreg | gboost
FEATURE_KEYS   = ['height', 'footprint', 'n_points', 'z_center', 'density']
MODEL_OUT      = 'runs/lidar_person_clf.joblib'

## 2. Load the recorded point clouds

In [ ]:
import glob, numpy as np
files = sorted(glob.glob(CLOUDS_DIR + '/*.npy'))
clouds = [np.load(f) for f in files]
print(len(clouds), 'frames; first', clouds[0].shape if clouds else None, '(x,y,z,intensity,ring)')

## 3. View a raw cloud inline (Plotly) - `pip install plotly`

In [ ]:
import numpy as np, plotly.graph_objects as go
p = clouds[VIEW_FRAME]
xy2 = p[:, 0]**2 + p[:, 1]**2
p = p[(xy2 < ROI_RADIUS**2) & (np.abs(p[:, 2]) < ROI_Z)]
if len(p) > VIEW_MAX_POINTS:
    p = p[np.random.choice(len(p), VIEW_MAX_POINTS, replace=False)]
go.Figure(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
    marker=dict(size=1.5, color=p[:, 2], colorscale='Viridis'))).update_layout(
    scene_aspectmode='data', height=600, title=f'L2 cloud frame {VIEW_FRAME}').show()

## 4. Preprocess + DBSCAN clusters, then view the clusters inline

In [ ]:
import os, numpy as np, plotly.graph_objects as go
from sklearn.cluster import DBSCAN
_OFF, _BASE = 1024, 4096

def fg_mask(pts, occupied, voxel):
    idx = np.floor(pts[:, :3] / voxel).astype(np.int64)
    keys = ((idx[:, 0] + _OFF) * _BASE + (idx[:, 1] + _OFF)) * _BASE + (idx[:, 2] + _OFF)
    return ~np.isin(keys, occupied)

def preprocess(pts):
    xy2 = pts[:, 0]**2 + pts[:, 1]**2
    pts = pts[(xy2 < ROI_RADIUS**2) & (np.abs(pts[:, 2]) < ROI_Z)]
    if SUBTRACT_BACKGROUND and os.path.exists(BACKGROUND_NPZ):
        bg = np.load(BACKGROUND_NPZ); pts = pts[fg_mask(pts, bg['occupied'], float(bg['voxel']))]
    if REMOVE_GROUND and len(pts):
        pts = pts[pts[:, 2] > np.percentile(pts[:, 2], GROUND_PERCENTILE)]
    return pts

def clusters_and_features(pts):
    feats, traces = [], []
    if len(pts) < DBSCAN_MIN_POINTS: return feats, traces
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_POINTS).fit_predict(pts[:, :3])
    for lab in range(labels.max() + 1):
        c = pts[labels == lab]
        if len(c) < MIN_CLUSTER_POINTS: continue
        bmin, bmax = c[:, :3].min(0), c[:, :3].max(0)
        h = float(bmax[2] - bmin[2]); fp = float(max(bmax[0] - bmin[0], bmax[1] - bmin[1]))
        vol = float(max(np.prod(bmax - bmin), 1e-6))
        feats.append({'height': h, 'footprint': fp, 'n_points': len(c),
                      'z_center': float(c[:, 2].mean()), 'density': len(c) / vol})
        traces.append(go.Scatter3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], mode='markers',
                                   marker=dict(size=2), name=f'cl{lab} h={h:.1f} n={len(c)}'))
    return feats, traces

pts = preprocess(clouds[VIEW_FRAME])
feats, traces = clusters_and_features(pts)
print(len(feats), 'clusters in frame', VIEW_FRAME)
if traces:
    go.Figure(traces).update_layout(scene_aspectmode='data', height=600,
        title=f'DBSCAN clusters (frame {VIEW_FRAME})').show()

## 5. Training data: harvested labels (real) or synthetic
The provided clouds have no people, so to train *now* use the synthetic labeled set. After calibration, point `HARVEST_FILE` at the JSONL `fuse.py` writes (real camera labels).

In [ ]:
import json, os, numpy as np
COLS = ['height', 'footprint', 'n_points', 'z_center', 'density']

def load_harvest(path):
    X, y = [], []
    for line in open(path):
        r = json.loads(line); dx, dy, dz = r['dims']
        f = {'height': dz, 'footprint': max(dx, dy), 'n_points': r.get('n_points', 100),
             'z_center': r['centroid'][2], 'density': r.get('density', 50)}
        X.append([f[k] for k in FEATURE_KEYS]); y.append(1 if r['label'] == 'person' else 0)
    return np.array(X), np.array(y)

def synth(n=600, seed=0):
    rng = np.random.default_rng(seed); h = n // 2
    P = np.stack([rng.normal(1.7, .12, h), rng.normal(.45, .08, h), rng.normal(300, 60, h),
                  rng.normal(.95, .1, h), rng.normal(120, 30, h)], 1)
    F = np.stack([rng.uniform(.4, 2.2, h), rng.uniform(.3, 1.5, h), rng.uniform(40, 500, h),
                  rng.uniform(.4, 1.2, h), rng.uniform(20, 200, h)], 1)
    idx = [COLS.index(k) for k in FEATURE_KEYS]
    return np.vstack([P, F])[:, idx], np.array([1] * h + [0] * h)

if USE_SYNTHETIC_LABELS or not os.path.exists(HARVEST_FILE):
    X, y = synth(seed=RANDOM_SEED); print('synthetic labels:', X.shape)
else:
    X, y = load_harvest(HARVEST_FILE); print('harvested labels:', X.shape)

## 6. Train + evaluate - `pip install scikit-learn joblib`

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

cw = 'balanced' if BALANCE_CLASSES else None
clf = {'random_forest': RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
          min_samples_leaf=MIN_SAMPLES_LEAF, class_weight=cw, random_state=RANDOM_SEED),
       'gboost': GradientBoostingClassifier(random_state=RANDOM_SEED),
       'logreg': LogisticRegression(max_iter=1000, class_weight=cw)}[CLASSIFIER]
model = make_pipeline(StandardScaler(), clf) if STANDARDIZE else clf
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SPLIT, random_state=RANDOM_SEED, stratify=y)
model.fit(Xtr, ytr)
print(classification_report(yte, model.predict(Xte)))
print(confusion_matrix(yte, model.predict(Xte)))
if CROSS_VALIDATE: print('CV acc:', cross_val_score(model, X, y, cv=CV_FOLDS).mean())

## 7. Save
Load in `classical.py` to replace the size heuristic with the learned classifier.

In [ ]:
if SAVE_MODEL:
    import os, joblib
    os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
    joblib.dump({'model': model, 'features': FEATURE_KEYS}, MODEL_OUT); print('saved ->', MODEL_OUT)